In [ ]:
import os
import sys

if sys.platform == "linux":
    os.environ["MUJOCO_GL"] = "egl"
    os.environ["PYOPENGL_PLATFORM"] = "egl"

# Create Simulation Environment

Scene with Franka robot + 3 objects: Bowl (receptacle), Mug (pick target), Bottle (place-next-to target).

In [ ]:
import mujoco
from mujoco import MjModel, MjData, MjSpec
from scipy.spatial.transform import Rotation as R
import numpy as np

from molmo_spaces.configs.robot_configs import FrankaRobotConfig
from molmo_spaces.robots.robot_views.franka_droid_view import FrankaDroidRobotView
from molmo_spaces.molmo_spaces_constants import get_procthor_10k_houses, get_robot_path
from molmo_spaces.robots.franka import FrankaRobot
from molmo_spaces.utils.lazy_loading_utils import install_scene_with_objects_and_grasps_from_path, install_uid

In [ ]:
houses = get_procthor_10k_houses(split="val")
house_xml_path = houses["val"][0]["base"]
install_scene_with_objects_and_grasps_from_path(house_xml_path)

spec = MjSpec.from_file(house_xml_path)

robot_config = FrankaRobotConfig(base_size=[0.5, 0.5, 0.75])
robot_file_path = get_robot_path(robot_config.name) / robot_config.robot_xml_path
robot_spec = MjSpec.from_file(str(robot_file_path))

FrankaRobot.add_robot_to_scene(
    robot_config,
    spec,
    robot_spec,
    prefix=robot_config.robot_namespace,
    pos=[6.8, 9.75],
    quat=R.from_euler("z", 90, degrees=True).as_quat(scalar_first=True),
)

spec.camera(robot_config.robot_namespace + "gripper/wrist_camera").resolution = [640, 360]
spec.body(robot_config.robot_namespace + "fr3_link0").add_camera(
    pos=[0.1, 0.57, 0.66],
    quat=[-0.3633, -0.1241, 0.4263, 0.8191],
    fovy=71.0,
    resolution=[640, 360],
    name="robot_0/exo_camera_1",
)

# --- Objects ---
# Bowl (receptacle for pick-and-place)
bowl_xml_path = install_uid("Bowl_3")
bowl_spec = MjSpec.from_file(str(bowl_xml_path))
receptacle_frame = spec.worldbody.add_frame(
    pos=[7.1, 10.2, 1.01],
    quat=R.from_euler("x", 90, degrees=True).as_quat(scalar_first=True)
)
receptacle_frame.attach_body(bowl_spec.worldbody.first_body(), prefix="place_receptacle/")

# Mug (for pick task)
mug_xml_path = install_uid("Mug_1")
mug_spec = MjSpec.from_file(str(mug_xml_path))
mug_frame = spec.worldbody.add_frame(
    pos=[6.9, 10.35, 1.01],
    quat=R.from_euler("x", 90, degrees=True).as_quat(scalar_first=True)
)
mug_frame.attach_body(mug_spec.worldbody.first_body(), prefix="mug_obj/")

# Bottle (for place-next-to task)
bottle_xml_path = install_uid("Bottle_1")
bottle_spec = MjSpec.from_file(str(bottle_xml_path))
bottle_frame = spec.worldbody.add_frame(
    pos=[7.2, 10.35, 1.01],
    quat=R.from_euler("x", 90, degrees=True).as_quat(scalar_first=True)
)
bottle_frame.attach_body(bottle_spec.worldbody.first_body(), prefix="bottle_obj/")

model: MjModel = spec.compile()
data = MjData(model)
view = FrankaDroidRobotView(data, robot_config.robot_namespace)

view.set_qpos_dict(robot_config.init_qpos)
mujoco.mj_forward(model, data)
for mg_id in view.move_group_ids():
    mg = view.get_move_group(mg_id)
    mg.ctrl = mg.noop_ctrl
mujoco.mj_forward(model, data)

renderer = mujoco.Renderer(model, 360, 640)
scene_option = mujoco.MjvOption()
scene_option.sitegroup = 0

def render():
    renderer.update_scene(data, camera="robot_0/exo_camera_1", scene_option=scene_option)
    exo_img = renderer.render()
    renderer.update_scene(data, camera="robot_0/gripper/wrist_camera", scene_option=scene_option)
    cam_img = renderer.render()
    return {
        "exo_camera_1": exo_img,
        "wrist_camera": cam_img,
    }

print("Scene compiled with Bowl, Mug, and Bottle.")

## Sample render

This is what the policy sees!

In [ ]:
from PIL import Image

images = render()
stacked_img = np.hstack([images["exo_camera_1"], images["wrist_camera"]])
Image.fromarray(stacked_img)

# Load Model

In [ ]:
from huggingface_hub import snapshot_download
from olmo.eval.configure_real_robot import RealRobotVLAPolicy, RealRobotVLAPolicyConfig

ckpt_path = snapshot_download("allenai/MolmoBot-DROID")

class MockConfig:
    def __init__(self, policy_config):
        self.policy_config = policy_config

policy_config = RealRobotVLAPolicyConfig()
policy_config.checkpoint_path = ckpt_path
policy_config.action_type = "joint_pos"
policy_config.action_keys["arm"] = "joint_pos"

mock_config = MockConfig(policy_config)

policy = RealRobotVLAPolicy(config=mock_config, task_type="manipulation")

# Run Policy — Multi-Task Demo

Each task resets the simulation and policy, then runs for `episode_dur` seconds.  
Per-step logs print so you know it's not stuck (inference takes ~80s on CPU every 8 steps).

In [ ]:
import time
from moviepy.video.io.ImageSequenceClip import ImageSequenceClip
from IPython.display import Video


def run_task(task, episode_dur=6.6, policy_dt_ms=66, video_filename=None):
    """Run a single task episode with sim reset and per-step logging."""
    if video_filename is None:
        video_filename = task.replace(" ", "_")[:40] + ".mp4"

    # Reset simulation
    mujoco.mj_resetData(model, data)
    view.set_qpos_dict(robot_config.init_qpos)
    mujoco.mj_forward(model, data)
    for mg_id in view.move_group_ids():
        mg = view.get_move_group(mg_id)
        mg.ctrl = mg.noop_ctrl
    mujoco.mj_forward(model, data)

    # Reset policy (clears action buffer + obs history)
    policy.reset()

    total_steps = round(episode_dur * 1000 / policy_dt_ms)
    print(f"\n{'='*60}")
    print(f"Task: \"{task}\"")
    print(f"Steps: {total_steps} | episode_dur: {episode_dur}s")
    print(f"{'='*60}")

    frames = []
    t0 = time.time()

    for step in range(total_steps):
        step_t0 = time.time()

        jp = view.get_move_group("arm").joint_pos
        gripper_input = view.get_move_group("gripper").joint_pos
        obs = {
            "task": task,
            "qpos": {
                "arm": jp,
                "gripper": gripper_input,
            },
            **render()
        }

        frame = np.hstack([obs["exo_camera_1"], obs["wrist_camera"]])
        frames.append(frame)

        # Log before get_action — inference happens every 8 steps
        is_inference_step = (step % 8 == 0) or len(policy.action_buffer) == 0 or policy.buffer_index >= len(policy.action_buffer)
        if is_inference_step:
            print(f"  Step {step+1}/{total_steps} | {time.time()-t0:.1f}s elapsed | running inference (~80s on CPU)...", end="", flush=True)

        action = policy.get_action(obs)

        if is_inference_step:
            print(f" done ({time.time()-step_t0:.1f}s)")
        else:
            gripper_val = action.get("gripper", [0])[0]
            print(f"  Step {step+1}/{total_steps} | {time.time()-t0:.1f}s elapsed | gripper: {gripper_val:.0f} | step: {time.time()-step_t0:.1f}s")

        for mg_id in action.keys():
            view.get_move_group(mg_id).ctrl = action[mg_id]

        mujoco.mj_step(model, data, nstep=policy_dt_ms // round(model.opt.timestep * 1000))

    elapsed = time.time() - t0
    print(f"Done! {total_steps} steps in {elapsed:.1f}s ({elapsed/60:.1f} min)")

    video = ImageSequenceClip(frames, fps=15)
    video.write_videofile(video_filename, audio=False, logger=None)
    print(f"Saved: {video_filename}")
    return Video(video_filename)

## Phase 1: Smoke Tests (5 steps each, ~2 min per task)

Verify scene, policy, and physics work before committing to full runs.

In [ ]:
run_task("put the salt shaker in the bowl", episode_dur=0.33, video_filename="smoke_pick_and_place.mp4")

In [ ]:
run_task("pick up the mug", episode_dur=0.33, video_filename="smoke_pick.mp4")

In [ ]:
run_task("place the bottle next to the bowl", episode_dur=0.33, video_filename="smoke_place_next_to.mp4")

## Phase 2: Full Runs (100 steps each, ~20 min per task)

Only run these after smoke tests pass. Total: ~60 min for all 3 tasks.

### Task 1: Pick and Place
"put the salt shaker in the bowl"

In [ ]:
run_task("put the salt shaker in the bowl", episode_dur=6.6, video_filename="task1_pick_and_place.mp4")

### Task 2: Pick
"pick up the mug"

In [ ]:
run_task("pick up the mug", episode_dur=6.6, video_filename="task2_pick.mp4")

### Task 3: Place Next To
"place the bottle next to the bowl"

In [ ]:
run_task("place the bottle next to the bowl", episode_dur=6.6, video_filename="task3_place_next_to.mp4")

### Custom Task
Edit the prompt below to try your own task.

In [ ]:
# run_task("your task here", episode_dur=6.6)